In [ ]:
import fitz
import re
import os
import mimetypes
import pandas as pd

from pathlib import Path

from exchangelib import (
    Credentials,
    Configuration,
    Account,
    Message,
    HTMLBody,
    FileAttachment
)


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_PDF = "GeneratedReport_43520.pdf"

EMPLOYEE_EXCEL = r"C:\Users\mturky\Documents\send.csv"

OUTPUT_DIR = Path("medical_cards_output")

# Excel column names
ID_COLUMN = "ID"
EMAIL_COLUMN = "Email"


# ============================================================
# EMAIL CONFIGURATION
# ============================================================

MAIL_SERVER = "mail.cne.com.eg"

MAIL_USERNAME = "mturky"

MAIL_ADDRESS = "mturky@cne.com.eg"

# Better: set this as an environment variable
MAIL_PASSWORD = "moh@mmed"


# ============================================================
# REGEX
# ============================================================

# Examples:
#
# 5110536728E0
# 5110536728W1
# 5110536728D1
# 5110536728S1

ID_PATTERN = re.compile(
    r"\b(\d{10}[A-Z]\d+)\b",
    re.IGNORECASE
)


# Example:
#
# Name: Ahmed Mohamed Ali

NAME_PATTERN = re.compile(
    r"Name\s*:\s*(.+)",
    re.IGNORECASE
)


# ============================================================
# PDF FUNCTIONS
# ============================================================

def extract_card_id(text):

    match = ID_PATTERN.search(text)

    if match:
        return match.group(1).upper()

    return None


def extract_name(text):

    match = NAME_PATTERN.search(text)

    if not match:
        return None

    name = match.group(1).strip()

    # Only take the first line
    name = name.split("\n")[0].strip()

    # Remove characters that are invalid in filenames
    name = re.sub(
        r'[<>:"/\\|?*]',
        '',
        name
    )

    # Remove duplicate spaces
    name = re.sub(
        r'\s+',
        ' ',
        name
    )

    return name


# ============================================================
# SPLIT PDF
# ============================================================

def split_medical_cards():

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    source = fitz.open(INPUT_PDF)

    # print("=" * 70)
    # print("PASS 1 - FINDING EMPLOYEES")
    # print("=" * 70)

    employees = {}

    # --------------------------------------------------------
    # PASS 1
    # Find employee cards
    # --------------------------------------------------------

    for page_number, page in enumerate(
        source,
        start=1
    ):

        text = page.get_text("text")

        card_id = extract_card_id(text)

        if not card_id:
            continue

        # Employee card
        if not re.search(
            r"E\d+$",
            card_id,
            re.IGNORECASE
        ):
            continue

        family_id = card_id[:10]

        employee_name = extract_name(text)

        if employee_name:

            employees[family_id] = employee_name

            # print(
            #     f"{family_id} -> {employee_name}"
            # )

        else:

            print(
                f"WARNING: Name not found "
                f"for {card_id} "
                f"on page {page_number}"
            )

    print()
    # print(
    #     f"Employees found: {len(employees)}"
    # )

    # --------------------------------------------------------
    # PASS 2
    # Split every card
    # --------------------------------------------------------

    # print()
    # print("=" * 70)
    # print("PASS 2 - SPLITTING CARDS")
    # print("=" * 70)

    processed = 0
    failed = []

    for page_number, page in enumerate(
        source,
        start=1
    ):

        text = page.get_text("text")

        card_id = extract_card_id(text)

        if not card_id:

            failed.append(
                (page_number, "No card ID")
            )

            # print(
            #     f"WARNING: No ID "
            #     f"on page {page_number}"
            # )

            continue

        family_id = card_id[:10]

        employee_name = employees.get(
            family_id
        )

        if not employee_name:

            failed.append(
                (
                    page_number,
                    f"No employee for {card_id}"
                )
            )

            # print(
            #     f"WARNING: Employee not found "
            #     f"for {card_id}"
            # )

            continue

        # ----------------------------------------------------
        # Check "يتحمل" on employee card
        # ----------------------------------------------------

        # if re.search(
        #     r"E\d+$",
        #     card_id,
        #     re.IGNORECASE
        # ):

        #     if "يتحمل" in text:

        #         print()
        #         print(
        #             "*** FOUND 'يتحمل' ***"
        #         )

        #         print(
        #             f"Employee ID   : {family_id}"
        #         )

        #         print(
        #             f"Employee Name : {employee_name}"
        #         )

        #         print(
        #             f"Card ID       : {card_id}"
        #         )

        #         print(
        #             f"Page          : {page_number}"
        #         )

        #         print(
        #             "************************"
        #         )
        #         print()

        # ----------------------------------------------------
        # Create employee folder
        # ----------------------------------------------------

        folder_name = (
            # f"{family_id} - {employee_name}"
            f"{family_id}"

        )

        family_folder = (
            OUTPUT_DIR / folder_name
        )

        family_folder.mkdir(
            parents=True,
            exist_ok=True
        )

        # ----------------------------------------------------
        # Save card
        # ----------------------------------------------------

        output_file = (
            family_folder / f"{card_id}.pdf"
        )

        new_pdf = fitz.open()

        new_pdf.insert_pdf(
            source,
            from_page=page_number - 1,
            to_page=page_number - 1
        )

        new_pdf.save(output_file)

        new_pdf.close()

        processed += 1

        # print(
        #     f"{card_id} -> "
        #     f"{folder_name}"
        # )

    source.close()

    # print()
    # print(
    #     f"Cards processed: {processed}"
    # )

    return employees


# ============================================================
# EMAIL FUNCTION
# ============================================================

def sendMail(
    subject,
    body,
    to,
    cc=None,
    attachments=None
):

    if not MAIL_PASSWORD:
        raise RuntimeError(
            "MAIL_PASSWORD environment variable "
            "is not set."
        )

    credentials = Credentials(
        username=MAIL_USERNAME,
        password=MAIL_PASSWORD
    )

    config = Configuration(
        server=MAIL_SERVER,
        credentials=credentials
    )

    myaccount = Account(
        primary_smtp_address=MAIL_ADDRESS,
        credentials=credentials,
        config=config
    )

    m = Message(
        account=myaccount,
        subject=f"[Auto] {subject}",
        body=HTMLBody(body),
        to_recipients=to,
        cc_recipients=cc,
        bcc_recipients=[]
    )

    if attachments:

        for file_path in attachments:

            file_path = Path(file_path)

            # Determine MIME type
            content_type, _ = mimetypes.guess_type(
                file_path.name
            )

            if not content_type:
                content_type = "application/octet-stream"

            att = FileAttachment(
                name=file_path.name,
                content_type=content_type,
                is_inline=False,
                is_contact_photo=False,
                content=file_path.read_bytes()
            )

            m.attach(att)

    m.send()


# ============================================================
# READ EMPLOYEE EXCEL
# ============================================================

def load_employees():

    df = pd.read_csv(
        EMPLOYEE_EXCEL,
        dtype=str
    )

    # Remove spaces from column names
    df.columns = df.columns.str.strip()

    if ID_COLUMN not in df.columns:
        raise ValueError(
            f"Column '{ID_COLUMN}' not found in Excel."
        )

    if EMAIL_COLUMN not in df.columns:
        raise ValueError(
            f"Column '{EMAIL_COLUMN}' not found in Excel."
        )

    # Keep IDs as strings
    df[ID_COLUMN] = (
        df[ID_COLUMN]
        .astype(str)
        .str.strip()
    )

    df[EMAIL_COLUMN] = (
        df[EMAIL_COLUMN]
        .astype(str)
        .str.strip()
    )

    # In case Excel contains IDs like:
    #
    # 5110536728.0
    #
    df[ID_COLUMN] = (
        df[ID_COLUMN]
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )

    return df


# ============================================================
# SEND EMAILS
# ============================================================

def send_employee_emails():

    df = load_employees()

    print()
    print("=" * 70)
    print("SENDING EMAILS")
    print("=" * 70)

    sent = 0
    missing_email = []
    missing_folder = []
    errors = []

    # --------------------------------------------------------
    # Loop through every employee in Excel
    # --------------------------------------------------------

    for _, row in df.iterrows():

        employee_id = row[ID_COLUMN]
        email = row[EMAIL_COLUMN]

        if not employee_id:
            continue

        if not email or email.lower() == "nan":

            print(
                f"WARNING: No email for "
                f"{employee_id}"
            )

            missing_email.append(
                employee_id
            )

            continue

        # ----------------------------------------------------
        # Find employee folder
        #
        # Example:
        #
        # 5110536728 - Ahmed Mohamed Ali
        # ----------------------------------------------------

        folders = list(
            OUTPUT_DIR.glob(
                f"{employee_id}"
            )
        )

        if not folders:

            print(
                f"WARNING: No card folder for "
                f"{employee_id}"
            )

            missing_folder.append(
                employee_id
            )

            continue

        if len(folders) > 1:

            print(
                f"WARNING: Multiple folders found "
                f"for {employee_id}"
            )

            errors.append(
                (
                    employee_id,
                    "Multiple folders"
                )
            )

            continue

        employee_folder = folders[0]

        # ----------------------------------------------------
        # Get all PDF cards
        # ----------------------------------------------------

        attachments = sorted(
            employee_folder.glob("*.pdf")
        )

        if not attachments:

            print(
                f"WARNING: No cards in "
                f"{employee_folder}"
            )

            errors.append(
                (
                    employee_id,
                    "No PDF cards"
                )
            )

            continue

        # ----------------------------------------------------
        # Get employee name from folder
        # ----------------------------------------------------

        # employee_name = (
        #     employee_folder.name
        #     .split(" - ", 1)[1]
        # )

        # ----------------------------------------------------
        # Email body
        # ----------------------------------------------------

        body = f"""
        <html>
        <body>

        <p>Dear Collegue,</p>

        <p>
        Please find attached your medical cards
        and the medical cards of your registered
        family members.
        </p>

        <p>
        <b>Employee ID:</b> {employee_id}
        </p>

        <p>
        Regards,<br>
        CNE
        </p>

        </body>
        </html>
        """

        print()
        print(
            f"Sending to: {email}"
        )

        # print(
        #     f"Email: {email}"
        # )

        print(
            f"Cards: {len(attachments)}"
        )

        try:

            sendMail(
                subject="Medical Cards",
                body=body,
                # to=[email]
                to = "mturky@cne.com.eg",
                attachments=attachments
            )

            sent += 1

            print(
                ">>> SENT"
            )

        except Exception as e:

            print(
                f">>> ERROR: {e}"
            )

            errors.append(
                (
                    employee_id,
                    str(e)
                )
            )

    # ========================================================
    # SUMMARY
    # ========================================================

    print()
    print("=" * 70)
    print("EMAIL SUMMARY")
    print("=" * 70)

    print(
        f"Emails sent       : {sent}"
    )

    print(
        f"Missing emails    : {len(missing_email)}"
    )

    print(
        f"Missing folders   : {len(missing_folder)}"
    )

    print(
        f"Errors            : {len(errors)}"
    )

    if missing_email:

        print()
        print("Employees without email:")

        for employee_id in missing_email:
            print(employee_id)

    if missing_folder:

        print()
        print("Employees without cards:")

        for employee_id in missing_folder:
            print(employee_id)

    if errors:

        print()
        print("Errors:")

        for employee_id, error in errors:
            print(
                f"{employee_id}: {error}"
            )


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    # 1. Split PDF
    split_medical_cards()

    # 2. Send emails
    # send_employee_emails()